In [3]:
#File 1: train_model.py (Run this first to save your model)


import pandas as pd
import numpy as np
import joblib
from sklearn.tree import DecisionTreeClassifier

# 1. Load Titanic dataset
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# 2. Preprocess data
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Fare'] = df['Fare'].fillna(df['Fare'].median())
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
X = df[features]
y = df['Survived']

# 3. Train Decision Tree Classifier
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X, y)

# 4. Save model using joblib
joblib.dump(model, 'titanic_model.pkl')
print("Model trained and saved successfully as 'titanic_model.pkl'!")

Model trained and saved successfully as 'titanic_model.pkl'!


In [5]:
%%writefile app.py
#File 2: app.py (Interactive Streamlit Web App)

import streamlit as st
import joblib
import numpy as np

# Page configuration
st.set_page_config(page_title="Titanic Survival Predictor", page_icon="Ġ", layout="centered")

st.title("Ġ Titanic Survival Prediction App")
st.write("Adjust the passenger features in the sidebar to get real-time survival predictions.")

# 1. Load the pre-trained model with joblib
@st.cache_resource
def load_model():
    return joblib.load('titanic_model.pkl')

model = load_model()

# 2. Sidebar inputs (Sliders, Selectboxes)
st.sidebar.header("Passenger Features")

pclass = st.sidebar.selectbox("Passenger Class (Pclass)", [1, 2, 3], index=2)
sex = st.sidebar.selectbox("Gender", ["Male", "Female"])
age = st.sidebar.slider("Age", min_value=1, max_value=80, value=25)
sibsp = st.sidebar.number_input("Siblings/Spouses Aboard (SibSp)", min_value=0, max_value=10, value=0)
parch = st.sidebar.number_input("Parents/Children Aboard (Parch)", min_value=0, max_value=10, value=0)
fare = st.sidebar.slider("Fare Paid ($)", min_value=0.0, max_value=500.0, value=32.0)

# Convert categorical input to numerical
sex_encoded = 1 if sex == "Female" else 0

# 3. Interactive Prediction Display
if st.button("Predict Survival Status"):
    input_features = np.array([[pclass, sex_encoded, age, sibsp, parch, fare]])
    prediction = model.predict(input_features)[0]
    probability = model.predict_proba(input_features)[0]

    st.subheader("Prediction Result:")
    if prediction == 1:
        st.success(f"Ăā **Survived** (Confidence: {probability[1]*100:.1f}%)")
    else:
        st.error(f"Ą **Did Not Survive** (Confidence: {probability[0]*100:.1f}%)")

Writing app.py
